# 06. 모니터링 & AOP 패턴 — 비즈니스 로직에서 잡일을 분리하기

`난이도: 중급` · `소요: 25분` · `사전 지식:` [01. 프로젝트 개요](./01_project_overview.ipynb) `완료`

---

## 목차
1. [환경 설정](#1.-환경-설정)
2. [AOP 개념](#2.-AOP-(관점-지향-프로그래밍)-개념) — "요리사가 요리만 하게 해주는 시스템"
3. [@measure_time 라인별 분석](#3.-@measure_time-데코레이터-라인별-분석) — 3중 중첩, functools.wraps, perf_counter
4. [Prometheus 메트릭 타입](#4.-Prometheus-메트릭-타입-설명) — Histogram(키재기), Counter(만보기), Gauge(온도계)
5. [MetricsCollector 분석](#5.-MetricsCollector-분석) — 싱글턴 패턴
6. [/metrics 엔드포인트 파싱](#6.-/metrics-엔드포인트-파싱) — Prometheus 텍스트 형식
7. [커스텀 데코레이터 만들기](#7.-커스텀-데코레이터-직접-만들기) — retry 패턴 실습
8. [정리](#8.-정리-및-요약)

---

### 학습 목표
- AOP가 **왜 필요한지** 실생활 비유로 이해합니다
- `@measure_time` 데코레이터가 **어떻게** elapsed_ms를 주입하는지 한 줄씩 따라갑니다
- Prometheus 메트릭 3가지 타입을 **비유로** 구분할 수 있게 됩니다
- 직접 **retry 데코레이터**를 만들어봅니다

> **Tip** — 브라우저에서 `http://localhost:8000/metrics`를 열면 Prometheus 메트릭을 직접 볼 수 있습니다.

> **관련** — Prometheus UI(`http://localhost:9090`)에서 PromQL로 쿼리를 실습할 수 있습니다.

---

#### 노트북 네비게이션
| 이전 | 현재 | 다음 |
|------|------|------|
| [05. 벤치마크 & 시각화](./05_benchmark_and_visualization.ipynb) | **06. 모니터링 & AOP** | [07. 고급 패턴](./07_advanced_patterns.ipynb) |

**전체 학습 경로**: `01 개요` → `02 Redis` · `03 RabbitMQ` · `04 Kafka` → `05 벤치마크` → **`06 모니터링`** → `07 고급패턴` → `08 안정성` → `09 동시성` → `10 Saga`

---
## 1. 환경 설정

노트북에서 프로젝트 모듈을 import하려면 `sys.path`에 프로젝트 루트를 추가해야 합니다.
`httpx.AsyncClient`는 FastAPI 서버에 비동기 HTTP 요청을 보내기 위해 사용합니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=10.0)

---
## 2. AOP (관점 지향 프로그래밍) 개념

### AOP를 한 마디로 설명하면

> "본업에 집중할 수 있도록, 잡일은 자동으로 처리해주는 시스템"

실생활 비유로 이해해봅시다:


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 240" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">AOP (관점 지향 프로그래밍) 데코레이터 메커니즘</text> <g transform="translate(40, 80)"> <rect x="0" y="0" width="120" height="70" rx="8" fill="#E0F2FE" stroke="#0284C7" stroke-width="1.5" /> <text x="60" y="32" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#0369A1" text-anchor="middle">Client</text> <text x="60" y="52" font-family="-apple-system, sans-serif" font-size="10" fill="#0C4A6E" text-anchor="middle">API 호출</text> </g> <path d="M 160 115 L 245 115" stroke="#0284C7" stroke-width="2" /><polygon points="239,111 245,115 239,119" fill="#0284C7" /> <g transform="translate(250, 60)"> <rect x="0" y="0" width="280" height="110" rx="8" fill="#FEE2E2" stroke="#B91C1C" stroke-width="2" /> <text x="140" y="24" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#991B1B" text-anchor="middle">AOP Proxy (@measure_time)</text> <rect x="15" y="40" width="250" height="50" rx="4" fill="#FFFFFF" stroke="#FCA5A5" /> <text x="140" y="58" font-family="-apple-system, sans-serif" font-size="10" fill="#334155" text-anchor="middle">1. 시작 시간 기록 &amp; Prometheus 게이지 작동</text> <text x="140" y="76" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#991B1B" text-anchor="middle">2. 핵심 비즈니스 로직 호출 ➡️</text> </g> <path d="M 530 115 L 615 115" stroke="#B91C1C" stroke-width="2" /><polygon points="609,111 615,115 609,119" fill="#B91C1C" /> <g transform="translate(620, 80)"> <rect x="0" y="0" width="140" height="70" rx="8" fill="#DCFCE7" stroke="#15803D" stroke-width="1.5" /> <text x="70" y="32" font-family="-apple-system, sans-serif" font-size="12" font-weight="bold" fill="#166534" text-anchor="middle">Business Logic</text> <text x="70" y="52" font-family="-apple-system, sans-serif" font-size="10" fill="#14532D" text-anchor="middle">메시지 발행 / 처리</text> </g> <path d="M 620 135 L 530 135" stroke="#15803D" stroke-width="1.5" stroke-dasharray="3 3" /><polygon points="536,139 530,135 536,131" fill="#15803D" /> <path d="M 250 135 L 160 135" stroke="#0284C7" stroke-width="1.5" stroke-dasharray="3 3" /><polygon points="166,139 160,135 166,131" fill="#0284C7" /> <text x="390" y="200" font-family="-apple-system, sans-serif" font-size="10" fill="#7F1D1D" text-anchor="middle">3. 종료 시간 기록 ➡️ 소요 시간 산출 ➡️ Prometheus 메트릭 수집 및 클라이언트에 결과 반환</text> </svg> </div>


### AOP 용어 정리

Java Spring의 AOP 용어를 Python 데코레이터에 대응시키면:

| AOP 용어 | 의미 | Python 대응 |
|----------|------|------------|
| **Aspect** | 횡단 관심사 모듈 | `timer.py` 모듈 전체 |
| **Advice** | 실제 동작 코드 | `wrapper()` 함수 내부 |
| **Join Point** | 적용 가능한 시점 | 모든 async 함수 |
| **Pointcut** | 실제 적용 대상 | `@measure_time`이 붙은 함수 |
| **Weaving** | 관심사를 결합 | 데코레이터 적용 시점 |

In [ ]:
# AOP 적용 전후 비교: 실제 프로젝트 브로커에서 확인
from app.brokers.redis_broker import RedisBroker
import inspect

# publish 메서드에 데코레이터 메타정보가 있는지 확인
publish_method = RedisBroker.publish
broker_tag = getattr(publish_method, '_measure_time_broker', 'N/A')
op_tag = getattr(publish_method, '_measure_time_operation', 'N/A')

print(f"RedisBroker.publish:")
print(f"  _measure_time_broker    = {broker_tag}")
print(f"  _measure_time_operation = {op_tag}")
print(f"  __wrapped__ 존재 여부   = {hasattr(publish_method, '__wrapped__')}")

---
## 3. `@measure_time` 데코레이터 라인별 분석

이 프로젝트의 핵심 AOP 도구인 `@measure_time`을 한 줄씩 분석합니다.

### 3-1. 전체 구조: 3중 중첩 함수

```text
measure_time(broker, operation)   ← 팩토리 (설정값 캡처)
  └─ decorator(func)              ← 실제 데코레이터
       └─ wrapper(*args, **kwargs) ← 원본 함수를 감싸는 래퍼
```

**왜 3중 중첩인가?** `@measure_time(broker="redis")`처럼 인자를 받으려면,
첫 번째 호출이 설정값을 받고, 두 번째 호출이 함수를 받아야 하기 때문입니다.

In [ ]:
# 3중 중첩 구조 직접 확인
from app.monitoring.timer import measure_time

# 1단계: measure_time(...)은 decorator를 반환
decorator = measure_time(broker="test", operation="publish")
print(f"1단계 반환: {decorator}")
print(f"  타입: {type(decorator).__name__}")

# 2단계: decorator(func)는 wrapper를 반환
async def dummy_func():
    return {"status": "ok"}

wrapped = decorator(dummy_func)
print(f"\n2단계 반환: {wrapped}")
print(f"  __name__: {wrapped.__name__}")

### 3-2. `functools.wraps`는 왜 필요한가?

데코레이터를 적용하면 원본 함수가 `wrapper`로 대체됩니다.
`functools.wraps` 없이는 함수의 이름, docstring, 시그니처가 모두 사라집니다.

```python
@functools.wraps(func)  # func의 __name__, __doc__, __module__ 등을 wrapper에 복사
async def wrapper(*args, **kwargs):
    ...
```

이것이 없으면:
- `help(broker.publish)` → `wrapper`의 docstring이 표시됨
- FastAPI의 자동 문서 생성이 깨짐
- 디버깅 시 traceback에 `wrapper`만 표시됨

In [ ]:
import functools

# functools.wraps 없이 데코레이터 만들기
def bad_decorator(func):
    async def wrapper(*args, **kwargs):
        return await func(*args, **kwargs)
    return wrapper

# functools.wraps 있는 데코레이터
def good_decorator(func):
    @functools.wraps(func)
    async def wrapper(*args, **kwargs):
        return await func(*args, **kwargs)
    return wrapper

In [ ]:
async def my_publish(channel: str, message: str) -> dict:
    """메시지를 채널에 발행합니다."""
    return {"channel": channel}

bad_wrapped = bad_decorator(my_publish)
good_wrapped = good_decorator(my_publish)

print("=== functools.wraps 없음 ===")
print(f"  __name__: {bad_wrapped.__name__}")
print(f"  __doc__:  {bad_wrapped.__doc__}")

print("\n=== functools.wraps 있음 ===")
print(f"  __name__: {good_wrapped.__name__}")
print(f"  __doc__:  {good_wrapped.__doc__}")

### 3-3. `time.perf_counter()` vs `time.time()` 차이

| 함수 | 정밀도 | 단조성 | 용도 |
|------|--------|--------|------|
| `time.time()` | ~ms | NTP 보정으로 **뒤로 갈 수 있음** | 현재 시각(Wall Clock) |
| `time.perf_counter()` | ~ns | **단조 증가 보장** | 성능 측정 (경과 시간) |
| `time.monotonic()` | ~ms | 단조 증가 보장 | 타이머, 대기 시간 |

`@measure_time`이 `perf_counter()`를 사용하는 이유:
1. **나노초 수준 정밀도** - 메시지 발행은 0.5ms~5ms 범위이므로 높은 정밀도 필수
2. **단조 증가** - NTP 시간 보정으로 측정값이 음수가 되는 버그 방지
3. **OS 최고 해상도 클록 사용** - Linux에서 `clock_gettime(CLOCK_MONOTONIC)` 호출

In [ ]:
import time

# 정밀도 비교: 같은 연산을 두 방법으로 측정
for name, clock in [("time.time", time.time), ("perf_counter", time.perf_counter)]:
    deltas = []
    for _ in range(1000):
        a = clock()
        b = clock()
        deltas.append(b - a)
    avg_ns = sum(deltas) / len(deltas) * 1e9
    print(f"{name:15s}: 평균 해상도 = {avg_ns:.1f} ns")

print("\n-> perf_counter가 더 높은 해상도를 제공합니다")

### 3-4. dict 반환값에 `elapsed_ms` 주입하는 트릭

```python
if isinstance(result, dict):
    result["elapsed_ms"] = elapsed_ms
```

이 2줄이 핵심 트릭입니다:
- 원본 함수는 `{"broker": "redis", "channel": "test"}`를 반환
- 데코레이터가 자동으로 `"elapsed_ms": 1.234`를 추가
- API 응답에 **측정 코드 없이** 실행 시간이 포함됨

**설계 규칙**: 모든 브로커의 `publish()`, `subscribe()`가 `dict`를 반환하도록 `AbstractBroker`에서 강제합니다.

In [ ]:
# dict 주입 트릭 직접 확인: 메시지를 발행하고 응답 확인
r = await client.post("/redis/pubsub/publish", json={
    "content": "AOP elapsed_ms injection test",
    "metadata": {"notebook": "06"}
})
result = r.json()

print("API 응답:")
for key, value in result.items():
    marker = " <-- 데코레이터가 주입" if key == "elapsed_ms" else ""
    print(f"  {key}: {value}{marker}")

### 3-5. Prometheus 자동 기록

```python
if record_metric:
    if operation == "publish":
        metrics_collector.record_publish(broker, elapsed_ms)
    elif operation == "consume":
        metrics_collector.record_consume(broker, elapsed_ms)
```

`record_metric=True`(기본값)이면 Prometheus에 자동으로 기록됩니다.
벤치마크 내부 반복에서는 `record_metric=False`로 꺼서 메트릭 오염을 방지합니다.

**데코레이터 하나로 두 가지 관심사를 동시에 처리:**
1. 응답에 `elapsed_ms` 주입 (API 소비자용)
2. Prometheus 메트릭 기록 (운영 모니터링용)

In [ ]:
# 데코레이터 메타정보로 어떤 브로커에 적용되었는지 확인
from app.brokers.redis_broker import RedisBroker
from app.brokers.rabbitmq_broker import RabbitMQBroker
from app.brokers.kafka_broker import KafkaBroker

for cls in [RedisBroker, RabbitMQBroker, KafkaBroker]:
    for method_name in ['publish', 'subscribe']:
        method = getattr(cls, method_name, None)
        if method:
            b = getattr(method, '_measure_time_broker', '-')
            o = getattr(method, '_measure_time_operation', '-')
            print(f"{cls.__name__:20s}.{method_name:12s} -> broker={b}, op={o}")

---
## 4. Prometheus 메트릭 타입 설명

### 메트릭이란?

메트릭은 **시스템의 상태를 숫자로 표현한 것**입니다.
병원에서 환자의 상태를 체온, 혈압, 심박수로 파악하듯이,
서버의 상태도 처리량, 지연시간, 연결 수 등의 숫자로 파악합니다.

```text
  병원 모니터링:                    서버 모니터링:
  체온: 36.5°C                     지연시간: 0.5ms
  혈압: 120/80                     처리량: 1000 msg/sec
  심박수: 72                       연결 수: 3
  → 정상!                          → 정상!
  
  체온: 39.5°C ← 이상!             지연시간: 500ms ← 이상!
  → 알람 발생                      → 알람 발생 (Prometheus Alert)
```

### Prometheus의 3가지 메트릭 타입

이 프로젝트에서 사용하는 3가지 타입을 **실생활 비유**로 설명합니다.

### 4-1. Histogram (히스토그램) — "키 재기 부스"

학교 건강검진에서 학생들의 키를 측정하는 것과 같습니다.
단순히 평균만 구하는 것이 아니라, **구간별로 몇 명인지** 분류합니다.

```text
  키 측정 결과를 구간(버킷)별로 분류:
  
  150cm 이하:  ████  (4명)
  160cm 이하:  ████████████  (12명, 누적)
  170cm 이하:  ████████████████████  (20명, 누적)
  180cm 이하:  ██████████████████████████  (26명, 누적)
  +Inf:        ████████████████████████████  (28명, 전체)
  
  → "대부분 160~170cm 사이에 분포"라는 것을 알 수 있음
```

이 프로젝트의 Histogram은 **메시지 발행 지연시간**을 측정합니다:

```python
PUBLISH_LATENCY = Histogram(
    "broker_publish_latency_seconds",
    buckets=[0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
)
```

| 버킷 (초) | 밀리초 | 의미 |
|-----------|--------|------|
| 0.0005 | 0.5ms | Redis 수준 (초고속) |
| 0.001 | 1ms | Redis~RabbitMQ 경계 |
| 0.005 | 5ms | RabbitMQ 일반 범위 |
| 0.01 | 10ms | Kafka 일반 범위 |
| 0.05 | 50ms | 네트워크 지연 포함 |
| 0.1~1.0 | 100~1000ms | 느린 응답~타임아웃 |

> 버킷이 **로그 스케일**로 설계되어, 빠른 응답(0.5ms)과 느린 응답(1000ms) 모두를 세밀하게 구분합니다.

### 4-2. Counter (카운터) — "만보기"

만보기처럼 **항상 증가만 하는** 숫자입니다. 절대 감소하지 않습니다.

```text
  만보기:  0 → 100 → 250 → 500 → 1000 (계속 증가)
  
  Counter: 발행 0건 → 100건 → 250건 → 500건 → 1000건
  
  "지금까지 총 몇 건 발행했나?"에 답하는 메트릭
  Prometheus에서 rate()로 변환하면 "초당 몇 건?"도 계산 가능
```

### 4-3. Gauge (게이지) — "온도계"

온도계처럼 **올라갈 수도, 내려갈 수도** 있는 현재 값입니다.

```text
  온도계:  20°C → 25°C → 22°C → 18°C (오르락내리락)
  
  Gauge: 연결 3개 → 연결 5개 → 연결 2개 (브로커 연결 수)
  
  "지금 현재 연결된 브로커가 몇 개인가?"에 답하는 메트릭
```

### 3가지 타입 비교 요약

| | Histogram | Counter | Gauge |
|---|-----------|---------|-------|
| **비유** | 키 재기 부스 | 만보기 | 온도계 |
| **방향** | 관찰만 (분포 기록) | 증가만 | 증가+감소 |
| **질문** | "분포가 어떤가?" | "총 몇 건?" | "지금 몇 개?" |
| **프로젝트 예** | 발행 지연시간 | 총 발행 수 | 활성 연결 수 |
| **PromQL** | `histogram_quantile()` | `rate()` | 그대로 사용 |

In [ ]:
# Prometheus Histogram이 내부적으로 어떻게 동작하는지 확인
from app.monitoring.metrics import PUBLISH_LATENCY

# 버킷 경계값 확인
print("PUBLISH_LATENCY 버킷 경계값:")
for b in PUBLISH_LATENCY._kwargs.get('buckets', []):
    print(f"  {b:8.4f}초 = {b*1000:8.2f}ms")

print(f"\n-> +Inf 버킷이 자동 추가됨 (모든 값을 포함)")

### 4-2. Counter (카운터)

**단조 증가**하는 값입니다. 절대로 감소하지 않습니다.

```python
PUBLISH_COUNT = Counter(
    "broker_publish_total",
    "발행된 메시지 총 개수",
    labelnames=["broker"],
)
```

- `inc()` 호출 시 1 증가
- 서버 재시작 시 0으로 리셋
- Prometheus에서 `rate()`로 초당 발행률 계산 가능:
  ```promql
  rate(broker_publish_total{broker="redis"}[5m])
  ```

### 4-3. Gauge (게이지)

**현재 값**을 나타냅니다. 증가와 감소 모두 가능합니다.

```python
ACTIVE_CONNECTIONS = Gauge(
    "broker_active_connections",
    "현재 활성 연결 수",
    labelnames=["broker"],
)
```

- `inc()`: 연결 시 +1
- `dec()`: 해제 시 -1
- `set(value)`: 직접 설정
- 예: 현재 연결된 소비자 수, 대기 중인 메시지 수

In [ ]:
# Counter vs Gauge 동작 차이 확인
from prometheus_client import Counter, Gauge

# 테스트용 메트릭 생성
test_counter = Counter('test_notebook_counter', 'test', labelnames=['op'])
test_gauge = Gauge('test_notebook_gauge', 'test', labelnames=['op'])

# Counter: 증가만 가능
test_counter.labels(op='demo').inc()
test_counter.labels(op='demo').inc(5)
print(f"Counter 값: {test_counter.labels(op='demo')._value.get()}")

# Gauge: 증가/감소/설정 모두 가능
test_gauge.labels(op='demo').set(10)
test_gauge.labels(op='demo').dec(3)
print(f"Gauge 값:   {test_gauge.labels(op='demo')._value.get()}")

---
## 5. MetricsCollector 분석

`MetricsCollector`는 두 가지 역할을 합니다:

1. **Prometheus 메트릭 기록** - `record_publish()`, `record_consume()`
2. **인메모리 벤치마크 결과 저장** - `add_benchmark()`, `get_comparison()`, `get_history()`

### 싱글턴 패턴

```python
# metrics.py 모듈 하단
metrics_collector = MetricsCollector()
```

Python에서는 모듈이 한 번만 로드되므로, 이 한 줄로 싱글턴이 보장됩니다.
어디서 import하든 동일한 인스턴스를 참조합니다.

In [ ]:
# 싱글턴 확인: 두 군데서 import해도 같은 객체인가?
from app.monitoring.metrics import metrics_collector as mc1
from app.monitoring.timer import metrics_collector as mc2

print(f"metrics.py에서 import: id={id(mc1)}")
print(f"timer.py에서 import:   id={id(mc2)}")
print(f"동일 객체 여부: {mc1 is mc2}")

In [ ]:
# record_publish 내부 동작 분석
# latency_ms를 초로 변환하여 Histogram에 기록하는 이유:
# -> Prometheus 관례상 시간 단위는 '초'를 사용

from app.monitoring.metrics import metrics_collector

print("record_publish 동작 흐름:")
print("  1. PUBLISH_LATENCY.labels(broker).observe(latency_ms / 1000)")
print("     -> ms를 초로 변환하여 Histogram에 기록")
print("  2. PUBLISH_COUNT.labels(broker).inc()")
print("     -> 발행 카운터 1 증가")
print("  3. self.publish_history[broker].append(latency_ms)")
print("     -> 인메모리 리스트에 원본 ms값 보관")

# 현재 히스토리 상태 확인
history = metrics_collector.get_history()
print(f"\n현재 수집된 히스토리: {list(history.keys())}")

---
## 6. `/metrics` 엔드포인트 파싱

Prometheus는 텍스트 기반의 exposition format을 사용합니다.
FastAPI의 `/metrics` 엔드포인트가 이 형식으로 메트릭을 노출합니다.

### Prometheus 텍스트 형식 규칙

```text
# HELP <metric_name> <설명>           ← 메트릭 설명
# TYPE <metric_name> <타입>           ← histogram, counter, gauge
<metric_name>{label="value"} <값>     ← 실제 데이터
```

Histogram은 자동으로 3가지 시계열을 생성합니다:
- `_bucket{le="0.005"}` → 5ms 이하인 요청 수 (누적)
- `_sum` → 전체 지연시간 합계
- `_count` → 전체 요청 수

In [ ]:
# /metrics 엔드포인트에서 Prometheus 텍스트 형식 가져오기
r = await client.get("/metrics")
metrics_text = r.text

# broker_publish 관련 메트릭만 필터링
print("=== broker_publish 관련 메트릭 ===")
for line in metrics_text.split("\n"):
    if "broker_publish" in line:
        print(line)

In [ ]:
# Histogram 버킷 데이터 파싱: le(less than or equal) 이해
print("=== Histogram 버킷 읽는 법 ===")
print()
for line in metrics_text.split("\n"):
    if "publish_latency" in line and "bucket" in line:
        print(line)

print("\n해석: le=\"0.005\"의 값이 10이면,")
print("       5ms 이하로 완료된 요청이 누적 10건이라는 의미")

In [ ]:
# /monitoring/comparison - 벤치마크 비교 데이터 확인
r = await client.get("/monitoring/comparison")
comparison = r.json()
print("=== 벤치마크 비교 ===")

import json
print(json.dumps(comparison, indent=2, ensure_ascii=False))

In [ ]:
# /monitoring/history - 브로커별 발행 히스토리 확인
r = await client.get("/monitoring/history")
history = r.json()
print("=== 발행 히스토리 ===")

for broker, data in history.items():
    count = data.get('count', 0)
    avg = data.get('avg_ms', 0)
    last = data.get('last_10', [])
    print(f"\n{broker}:")
    print(f"  총 발행 수: {count}")
    print(f"  평균 지연:  {avg:.4f}ms")
    print(f"  최근 10건:  {last}")

---
## 7. 커스텀 데코레이터 직접 만들기

AOP 패턴을 이해했으니, 직접 **retry 데코레이터**를 만들어봅니다.
실패 시 자동으로 재시도하는 횡단 관심사를 분리합니다.

### 설계 요구사항
- `max_retries`: 최대 재시도 횟수
- `delay_seconds`: 재시도 간 대기 시간
- `exceptions`: 재시도할 예외 타입 튜플
- 재시도 로그 출력
- `functools.wraps` 적용 (필수!)

In [ ]:
import asyncio
import functools
from collections.abc import Callable

def retry(
    max_retries: int = 3,
    delay_seconds: float = 0.1,
    exceptions: tuple = (Exception,),
) -> Callable:
    """실패 시 자동 재시도하는 AOP 데코레이터."""
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_retries + 1):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    print(f"  [{attempt}/{max_retries}] {func.__name__} 실패: {e}")
                    if attempt < max_retries:
                        await asyncio.sleep(delay_seconds)
            raise last_exc
        return wrapper
    return decorator

In [ ]:
# retry 데코레이터 테스트: 3번 중 마지막에 성공
call_count = 0

@retry(max_retries=3, delay_seconds=0.05, exceptions=(ConnectionError,))
async def unreliable_publish(message: str) -> dict:
    """불안정한 발행 함수 (테스트용)."""
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"연결 실패 (시도 {call_count})")
    return {"status": "ok", "message": message}

result = await unreliable_publish("retry test")
print(f"\n최종 결과: {result}")
print(f"총 호출 횟수: {call_count}")

In [ ]:
# 데코레이터 조합: @retry + @measure_time 패턴
# 순서 중요: 안쪽 데코레이터부터 적용됨

# @retry          <-- 바깥: 재시도 로직
# @measure_time   <-- 안쪽: 시간 측정
# async def func  <-- 원본 함수

# 즉, retry가 measure_time(func())를 감싸므로:
# retry → measure_time → func 순서로 호출
# 재시도할 때마다 시간이 개별 측정됨

print("데코레이터 적용 순서:")
print("  @retry           # 3. 가장 바깥 (재시도 관리)")
print("  @measure_time    # 2. 중간 (시간 측정)")
print("  async def func   # 1. 원본 함수")
print("")
print("호출 체인: retry -> measure_time -> func")

---
## 8. 정리 및 요약

### 핵심 개념 정리

| 개념 | 설명 | 프로젝트 적용 |
|------|------|---------------|
| **AOP** | 횡단 관심사를 비즈니스 로직에서 분리 | `@measure_time` 데코레이터 |
| **functools.wraps** | 데코레이터 적용 시 원본 함수 메타데이터 보존 | 모든 데코레이터에 필수 적용 |
| **perf_counter** | 나노초 정밀도의 단조 증가 타이머 | 성능 측정에 사용 |
| **Histogram** | 값의 분포를 버킷으로 측정 | 지연시간 분포 추적 |
| **Counter** | 단조 증가하는 누적 값 | 발행/소비 총 개수 |
| **Gauge** | 현재 값 (증감 가능) | 활성 연결 수 |
| **싱글턴** | 모듈 레벨 인스턴스로 전역 상태 공유 | `metrics_collector` |

### 아키텍처 요약

```text
                ┌─────────────────────────────┐
                │     @measure_time (AOP)      │
                │  ┌────────────────────────┐  │
                │  │  elapsed_ms 주입        │  │
                │  │  Prometheus 메트릭 기록  │  │
                │  └───────────┬────────────┘  │
                └──────────────┼───────────────┘
                               │
         ┌─────────────────────┼─────────────────────┐
         │                     │                     │
   RedisBroker          RabbitMQBroker         KafkaBroker
    .publish()            .publish()            .publish()
         │                     │                     │
         └─────────────────────┼─────────────────────┘
                               │
                      MetricsCollector
                    ┌──────────┼──────────┐
                    │          │          │
              Histogram    Counter     Gauge
                    │          │          │
                    └──────────┼──────────┘
                               │
                      /metrics (Prometheus)
                      /monitoring/comparison
                      /monitoring/history
```

### 배운 점

1. **측정 코드는 비즈니스 코드에 없어야 한다** - `timer.py`에 모든 측정 로직 집중
2. **Prometheus 버킷은 도메인에 맞게 설계** - 브로커 지연시간 범위(0.5ms~1s)에 최적화
3. **데코레이터 조합 순서가 중요** - 안쪽부터 바깥으로 실행됨
4. **ms ↔ 초 변환 주의** - 내부는 ms, Prometheus는 초 단위

### 다음 단계
- Grafana 대시보드에서 이 메트릭들을 시각화해보세요
- PromQL로 `rate()`, `histogram_quantile()` 쿼리를 실습해보세요

In [ ]:
# 정리
await client.aclose()
print("노트북 06 완료!")